# cAPTure: model-ready graph materialization

This notebook converts the prepared five-second development windows into immutable, fold-specific graph shards. It does not train a model and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

Run **SMOKE → manual review → FULL_DEV**. Each packet becomes one directed edge. Parallel edges and self-loops are preserved, empty windows remain represented by gaps in `window_index`, and `decision_time` is the window end.

## 1. Mount Drive and prepare the environment

Push the current branch before opening Colab. This notebook installs the existing CPU data dependencies only; it does not install the training stack or run tests.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_graph_materialization_work")

REPOSITORY_ALREADY_PRESENT = PROJECT_ROOT.exists()
if not REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
if REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )

required_files = [
    "code/python/requirements-capture.txt",
    "code/python/utils/capture_graph_materialization.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_packet_schema_v1.yaml",
    "configs/capture_preprocessing_v1.yaml",
    "configs/capture_graph_materialization_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_materialization import run_capture_graph_materialization

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU graph-materialization environment is ready.")

## 2. Bind the approved inputs

These IDs bind materialization to the verified FULL_DEV preparation, the fold-local preprocessing audit, and the Stage-1 decision. If your Drive IDs differ, edit only the three ID constants. A run requires `stage1_decision.md` to contain `PASS_WITH_LIMITATIONS`.

Materializing both feature views may require several GiB on Drive. `/content` is used only for one fold/scenario at a time. Reduce `BATCH_SIZE` if RAM is tight; this does not change the scientific contract.

In [ ]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_RUN_ID = "20260919T004143_161396Z_preprocessing"
STRUCTURAL_RUN_ID = "20260928T145715_010326Z_graph_structural_full_dev"

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
MATERIALIZATION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_materialization_v1.yaml"
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
PREPROCESSING_AUDIT_DIR = (
    DRIVE_ROOT / "preprocessing_runs" / PREPROCESSING_RUN_ID
)
STAGE1_DECISION_PATH = (
    DRIVE_ROOT / "graph_pilot_runs" / STRUCTURAL_RUN_ID / "stage1_decision.md"
)
BATCH_SIZE = 100_000

required_inputs = [
    MANIFEST_PATH, PACKET_SCHEMA_PATH, PREPROCESSING_SCHEMA_PATH,
    MATERIALIZATION_CONFIG_PATH, PREPARED_RUN_DIR,
    PREPROCESSING_AUDIT_DIR, STAGE1_DECISION_PATH,
]
for required_path in required_inputs:
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")
stage1_decision_lines = STAGE1_DECISION_PATH.read_text().splitlines()
if "Status: **PASS_WITH_LIMITATIONS**." not in stage1_decision_lines:
    raise RuntimeError("Stage 1 is not recorded as PASS_WITH_LIMITATIONS.")

print("Prepared input:", PREPARED_RUN_DIR)
print("Preprocessing audit:", PREPROCESSING_AUDIT_DIR)
print("Stage-1 decision:", STAGE1_DECISION_PATH)
print("Local staging:", LOCAL_WORK_ROOT)

## 3. Run SMOKE

SMOKE materializes `train_empty_conn` and `train_dollar_char` under both fold preprocessors. Leave the ID as `None` for a new run. To resume after an interruption or Colab restart, paste the previously printed ID.

In [ ]:
SMOKE_RUN_ID = None  # Paste an existing ID here to resume.
if SMOKE_RUN_ID is None:
    SMOKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_materialization_smoke"
    )
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "graph_materialization_runs" / SMOKE_RUN_ID
print("SMOKE run ID:", SMOKE_RUN_ID)
print("SMOKE output:", SMOKE_OUTPUT_DIR)

SMOKE_RESULT = run_capture_graph_materialization(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    materialization_config_path=MATERIALIZATION_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
    stage1_decision_path=STAGE1_DECISION_PATH,
    output_dir=SMOKE_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="SMOKE",
    batch_size=BATCH_SIZE,
)

## 4. Review SMOKE

The cross-fold checks must pass. `feature_tensors_differ=True` is expected because each fold has a separately fitted preprocessor; topology, labels, source rows, feature names, and semantic node mapping must remain identical. Inspect shard counts, compressed size, runtime, and peak RAM before continuing.

In [ ]:
def materialization_table(run_dir):
    manifest = json.loads(
        (run_dir / "graph_materialization_manifest.json").read_text()
    )
    rows = []
    for fold in ("A", "B"):
        for scenario in manifest["scenarios"]:
            report = json.loads(
                (run_dir / f"fold_{fold}" / scenario
                 / "scenario_fold_report.json").read_text()
            )
            rows.append({
                "fold": fold,
                "scenario": scenario,
                "graphs": report["graphs"],
                "edges": report["edges"],
                "nodes": report["distinct_global_nodes"],
                "feature_dim": report["feature_dim"],
                "shards": report["shard_count"],
                "compressed_GiB": report["compressed_shard_bytes"] / 1024**3,
                "wall_seconds": report["wall_seconds"],
                "peak_RSS_GiB": (
                    report["peak_sampled_rss_bytes"] / 1024**3
                    if report["peak_sampled_rss_bytes"] is not None else None
                ),
            })
    return manifest, pd.DataFrame(rows)

smoke_manifest, smoke_table = materialization_table(SMOKE_OUTPUT_DIR)
display(smoke_table)
display(pd.DataFrame.from_dict(
    smoke_manifest["cross_fold_invariants"], orient="index"
))
print(json.dumps(smoke_manifest["totals"], indent=2))
print("Held-out scenarios accessed:",
      smoke_manifest["provenance"]["held_out_scenarios_accessed"])
print("Training performed:",
      smoke_manifest["provenance"]["training_performed"])

## 5. Manual approval for FULL_DEV

Set `APPROVE_FULL_DEV=True` only after the SMOKE manifest and every row above are plausible. Record a concrete note; it becomes part of the notebook evidence, while the immutable Stage-1 decision remains the scientific gate.

In [ ]:
APPROVE_FULL_DEV = False
SMOKE_REVIEW_NOTE = ""

if not APPROVE_FULL_DEV:
    raise RuntimeError(
        "Review SMOKE, then set APPROVE_FULL_DEV=True and record the note."
    )
if not SMOKE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SMOKE_REVIEW_NOTE before FULL_DEV.")
print("FULL_DEV materialization approved from SMOKE:", SMOKE_RUN_ID)
print(SMOKE_REVIEW_NOTE)

## 6. Run FULL_DEV

FULL_DEV writes both fold-specific feature views for all five development scenarios. Completed fold/scenario directories are checksum-validated when resuming. Keep the printed run ID.

In [ ]:
FULL_DEV_RUN_ID = None  # Paste an existing ID here to resume.
if FULL_DEV_RUN_ID is None:
    FULL_DEV_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_materialization_full_dev"
    )
FULL_DEV_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / FULL_DEV_RUN_ID
)
print("FULL_DEV run ID:", FULL_DEV_RUN_ID)
print("FULL_DEV output:", FULL_DEV_OUTPUT_DIR)

FULL_DEV_RESULT = run_capture_graph_materialization(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    materialization_config_path=MATERIALIZATION_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
    stage1_decision_path=STAGE1_DECISION_PATH,
    output_dir=FULL_DEV_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="FULL_DEV",
    batch_size=BATCH_SIZE,
)

## 7. Review FULL_DEV

Do not start training until every cross-fold invariant is present and true, the unique packet total matches preparation, and resource use is acceptable. The next implementation step will be the shard loader plus the frozen one-seed training contract; it will consume this run ID without rebuilding graphs.

In [ ]:
full_manifest, full_table = materialization_table(FULL_DEV_OUTPUT_DIR)
full_invariants = pd.DataFrame.from_dict(
    full_manifest["cross_fold_invariants"], orient="index"
)
display(full_table)
display(full_invariants)
print(json.dumps(full_manifest["totals"], indent=2))
print("Compressed shards (GiB):",
      full_manifest["totals"]["compressed_shard_bytes"] / 1024**3)
print("Manifest:",
      FULL_DEV_OUTPUT_DIR / "graph_materialization_manifest.json")

if not full_invariants["topology_and_labels_identical"].all():
    raise RuntimeError("Cross-fold topology or label invariants failed.")
if full_manifest["provenance"]["held_out_scenarios_accessed"]:
    raise RuntimeError("A held-out scenario was accessed.")
if full_manifest["provenance"]["training_performed"]:
    raise RuntimeError("Training unexpectedly occurred during materialization.")